# Simple MLP Training (Direct Dataset Files)

This notebook is basic and direct.

It uses these files as datasets:
- `/data/top_dataset.csv`
- `/data/bottom_dataset.csv`

Classical evaluation settings are kept:
- `RobustScaler`
- `StratifiedKFold(n_splits=10, shuffle=True, random_state=90483257)`
- `cross_val_predict`
- metrics: `accuracy`, `macro_f1`, `balanced_accuracy`

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

In [4]:
TOP_FILE = "../data/top_dataset.csv"
BOTTOM_FILE = "../data/bottom_dataset.csv"

## Top File: Load CSV, then Train on ALL Listed Datasets

In [12]:
top_df = pd.read_csv(TOP_FILE)

print("Top CSV length:", len(top_df))
display(top_df.head(10))

print("\nLoading and preparing all top datasets...")

Top CSV length: 54


,dataset,target_column,rows,features,classes,class_imbalance,missing_values_count,missing_values_ratio,numeric_features,categorical_features,feature_type
0,agaricus_lepiota.tsv.gz,target,8145,22,2,0.000729,0,0,22,0,numeric
1,allbp.tsv.gz,target,3772,29,3,0.583682,0,0,29,0,numeric
2,allhyper.tsv.gz,target,3771,29,4,0.697607,0,0,29,0,numeric
3,allhypo.tsv.gz,target,3770,29,3,0.522510,0,0,29,0,numeric
4,allrep.tsv.gz,target,3772,29,4,0.685706,0,0,29,0,numeric
5,analcatdata_authorship.tsv.gz,target,841,70,4,0.062548,0,0,70,0,numeric
6,analcatdata_bankruptcy.tsv.gz,target,50,6,2,0.000000,0,0,6,0,numeric
7,analcatdata_creditscore.tsv.gz,target,100,6,2,0.105800,0,0,6,0,numeric
8,analcatdata_cyyoung9302.tsv.gz,target,92,10,2,0.172259,0,0,10,0,numeric
9,analcatdata_lawsuit.tsv.gz,target,264,4,2,0.366420,0,0,4,0,numeric



Loading and preparing all top datasets...


## Top Dataset: Explicit MLP Code

In [13]:
# MLP training loop for all top datasets
top_results = []

for idx, row in top_df.iterrows():
    dataset_name = row["dataset"]
    dataset_path = f"../data/{dataset_name}"
    
    try:
        # Load dataset
        data = pd.read_csv(dataset_path, compression="gzip", sep="\t")
        label_col = "class" if "class" in data.columns else "target"
        
        X = data.drop(columns=[label_col]).values.astype(float)
        y = data[label_col].values
        
        # MLP training
        mlp = MLPClassifier(
            hidden_layer_sizes=(100,),
            activation="relu",
            solver="adam",
            max_iter=1000,
            early_stopping=True,
            random_state=324089
        )
        
        pipeline = make_pipeline(RobustScaler(), mlp)
        cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=90483257)
        pred = cross_val_predict(pipeline, X, y, cv=cv)
        
        # Metrics
        accuracy = accuracy_score(y, pred)
        macro_f1 = f1_score(y, pred, average="macro", zero_division=0)
        balanced_accuracy = balanced_accuracy_score(y, pred)
        
        top_results.append({
            "dataset": dataset_name,
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "balanced_accuracy": balanced_accuracy,
        })
        
        print(f"[{idx+1}/{len(top_df)}] {dataset_name}: BA={balanced_accuracy:.4f}")
    except Exception as e:
        print(f"[{idx+1}/{len(top_df)}] {dataset_name}: ERROR - {e}")

top_results_df = pd.DataFrame(top_results)
print(f"\nTop Results ({len(top_results_df)} datasets):")
display(top_results_df)

[1/54] agaricus_lepiota.tsv.gz: BA=0.9985
[2/54] allbp.tsv.gz: BA=0.4172
[3/54] allhyper.tsv.gz: BA=0.2499
[4/54] allhypo.tsv.gz: BA=0.4173
[5/54] allrep.tsv.gz: BA=0.2500
[6/54] analcatdata_authorship.tsv.gz: BA=0.9851
[7/54] analcatdata_bankruptcy.tsv.gz: BA=0.6200
[8/54] analcatdata_creditscore.tsv.gz: BA=0.4452
[9/54] analcatdata_cyyoung9302.tsv.gz: BA=0.4715
[10/54] analcatdata_lawsuit.tsv.gz: BA=0.5241
[11/54] ann_thyroid.tsv.gz: BA=0.8780
[12/54] appendicitis.tsv.gz: BA=0.6793
[13/54] backache.tsv.gz: BA=0.4710
[14/54] breast_cancer.tsv.gz: BA=0.5975
[15/54] chess.tsv.gz: BA=0.9808
[16/54] churn.tsv.gz: BA=0.7359
[17/54] clean1.tsv.gz: BA=0.7906
[18/54] clean2.tsv.gz: BA=0.9925
[19/54] coil2000.tsv.gz: BA=0.5005
[20/54] collins.tsv.gz: BA=0.3983
[21/54] confidence.tsv.gz: BA=0.2639
[22/54] corral.tsv.gz: BA=0.7333
[23/54] dermatology.tsv.gz: BA=0.9053
[24/54] dis.tsv.gz: BA=0.4997
[25/54] dna.tsv.gz: BA=0.9408
[26/54] hypothyroid.tsv.gz: BA=0.5028
[27/54] irish.tsv.gz: BA=0.8015

,dataset,accuracy,macro_f1,balanced_accuracy
0,agaricus_lepiota.tsv.gz,0.998527,0.998524,0.998496
1,allbp.tsv.gz,0.963680,0.454407,0.417236
2,allhyper.tsv.gz,0.972686,0.246539,0.249864
3,allhypo.tsv.gz,0.932361,0.463106,0.417266
4,allrep.tsv.gz,0.967126,0.245822,0.250000
5,analcatdata_authorship.tsv.gz,0.985731,0.984568,0.985131
6,analcatdata_bankruptcy.tsv.gz,0.620000,0.600673,0.620000
7,analcatdata_creditscore.tsv.gz,0.650000,0.393939,0.445205
8,analcatdata_cyyoung9302.tsv.gz,0.717391,0.452381,0.471521
9,analcatdata_lawsuit.tsv.gz,0.882576,0.525703,0.524060


In [20]:
# Rank and save top results
rank_metric = "accuracy"

top_ranked_df = top_results_df.sort_values(by=rank_metric, ascending=False).reset_index(drop=True)
top_ranked_df.insert(0, "rank", top_ranked_df.index + 1)

display(top_ranked_df)
top_ranked_df.to_csv("top_mlp_results_ranked.csv", index=False)
print("Top ranked results saved to: top_mlp_results_ranked.csv")

,rank,dataset,accuracy,macro_f1,balanced_accuracy
0,1,agaricus_lepiota.tsv.gz,0.998527,0.998524,0.998496
1,2,shuttle.tsv.gz,0.998466,0.829200,0.836691
2,3,mushroom.tsv.gz,0.997784,0.997782,0.997808
3,4,clean2.tsv.gz,0.996817,0.993880,0.992490
4,5,pendigits.tsv.gz,0.990357,0.990372,0.990384
5,6,texture.tsv.gz,0.989273,0.989271,0.989273
6,7,mofn_3_7_10.tsv.gz,0.988671,0.983211,0.974315
7,8,analcatdata_authorship.tsv.gz,0.985731,0.984568,0.985131
8,9,dis.tsv.gz,0.984093,0.495991,0.499731
9,10,ann_thyroid.tsv.gz,0.982500,0.902972,0.877983


Top ranked results saved to: top_mlp_results_ranked.csv


## Bottom File: Load CSV, then Train on ALL Listed Datasets

In [14]:
bottom_df = pd.read_csv(BOTTOM_FILE)

print("Bottom CSV length:", len(bottom_df))
display(bottom_df.head(10))

print("\nLoading and preparing all bottom datasets...")

Bottom CSV length: 49


,dataset,target_column,rows,features,classes,class_imbalance,missing_values_count,missing_values_ratio,numeric_features,categorical_features,feature_type
0,GAMETES_Epistasis_2_Way_1000atts_0.4H_EDM_1_ED...,target,1600,1000,2,0.000000,0,0,1000,0,numeric
1,GAMETES_Epistasis_2_Way_20atts_0.1H_EDM_1_1.ts...,target,1600,20,2,0.000000,0,0,20,0,numeric
2,GAMETES_Epistasis_2_Way_20atts_0.4H_EDM_1_1.ts...,target,1600,20,2,0.000000,0,0,20,0,numeric
3,GAMETES_Epistasis_3_Way_20atts_0.2H_EDM_1_1.ts...,target,1600,20,2,0.000000,0,0,20,0,numeric
4,GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_...,target,1600,20,2,0.000000,0,0,20,0,numeric
5,GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_...,target,1600,20,2,0.000000,0,0,20,0,numeric
6,Hill_Valley_with_noise.tsv.gz,target,1212,100,2,0.000000,0,0,100,0,numeric
7,Hill_Valley_without_noise.tsv.gz,target,1212,100,2,0.000049,0,0,100,0,numeric
8,analcatdata_aids.tsv.gz,target,50,4,2,0.000000,0,0,4,0,numeric
9,analcatdata_asbestos.tsv.gz,target,83,3,2,0.005879,0,0,3,0,numeric



Loading and preparing all bottom datasets...


## Bottom Dataset: Explicit MLP Code

In [15]:
# MLP training loop for all bottom datasets
bottom_results = []

for idx, row in bottom_df.iterrows():
    dataset_name = row["dataset"]
    dataset_path = f"../data/{dataset_name}"
    
    try:
        # Load dataset
        data = pd.read_csv(dataset_path, compression="gzip", sep="\t")
        label_col = "class" if "class" in data.columns else "target"
        
        X = data.drop(columns=[label_col]).values.astype(float)
        y = data[label_col].values
        
        # MLP training
        mlp = MLPClassifier(
            hidden_layer_sizes=(100,),
            activation="relu",
            solver="adam",
            max_iter=1000,
            early_stopping=True,
            random_state=324089
        )
        
        pipeline = make_pipeline(RobustScaler(), mlp)
        cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=90483257)
        pred = cross_val_predict(pipeline, X, y, cv=cv)
        
        # Metrics
        accuracy = accuracy_score(y, pred)
        macro_f1 = f1_score(y, pred, average="macro", zero_division=0)
        balanced_accuracy = balanced_accuracy_score(y, pred)
        
        bottom_results.append({
            "dataset": dataset_name,
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "balanced_accuracy": balanced_accuracy,
        })
        
        print(f"[{idx+1}/{len(bottom_df)}] {dataset_name}: BA={balanced_accuracy:.4f}")
    except Exception as e:
        print(f"[{idx+1}/{len(bottom_df)}] {dataset_name}: ERROR - {e}")

bottom_results_df = pd.DataFrame(bottom_results)
print(f"\nBottom Results ({len(bottom_results_df)} datasets):")
display(bottom_results_df)

[1/49] GAMETES_Epistasis_2_Way_1000atts_0.4H_EDM_1_EDM_1_1.tsv.gz: BA=0.4831
[2/49] GAMETES_Epistasis_2_Way_20atts_0.1H_EDM_1_1.tsv.gz: BA=0.4962
[3/49] GAMETES_Epistasis_2_Way_20atts_0.4H_EDM_1_1.tsv.gz: BA=0.5931
[4/49] GAMETES_Epistasis_3_Way_20atts_0.2H_EDM_1_1.tsv.gz: BA=0.5225
[5/49] GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_50_EDM_2_001.tsv.gz: BA=0.5444
[6/49] GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_75_EDM_2_001.tsv.gz: BA=0.6044
[7/49] Hill_Valley_with_noise.tsv.gz: BA=0.5231
[8/49] Hill_Valley_without_noise.tsv.gz: BA=0.4826
[9/49] analcatdata_aids.tsv.gz: BA=0.4400
[10/49] analcatdata_asbestos.tsv.gz: BA=0.7512
[11/49] analcatdata_boxing1.tsv.gz: BA=0.5211
[12/49] analcatdata_boxing2.tsv.gz: BA=0.5813
[13/49] analcatdata_dmft.tsv.gz: BA=0.1892
[14/49] analcatdata_germangss.tsv.gz: BA=0.3350
[15/49] analcatdata_happiness.tsv.gz: BA=0.3333
[16/49] bupa.tsv.gz: BA=0.6195
[17/49] calendarDOW.tsv.gz: BA=0.4521
[18/49] cloud.tsv.gz: BA=0.3289
[19/49] connect_4.tsv.gz: BA

c:\Users\EYONGEBOB\Desktop\PMLB Project\.venv\lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 8 members, which is less than n_splits=10.
  warnings.warn(


[37/49] poker.tsv.gz: BA=0.6348
[38/49] postoperative_patient_data.tsv.gz: BA=0.4844
[39/49] profb.tsv.gz: BA=0.5022
[40/49] saheart.tsv.gz: BA=0.6374
[41/49] schizo.tsv.gz: BA=0.3267
[42/49] segmentation.tsv.gz: BA=0.9074
[43/49] sonar.tsv.gz: BA=0.7624
[44/49] tae.tsv.gz: BA=0.3675
[45/49] tic_tac_toe.tsv.gz: BA=0.5306
[46/49] vowel.tsv.gz: BA=0.4444
[47/49] wine_quality_red.tsv.gz: BA=0.2576


c:\Users\EYONGEBOB\Desktop\PMLB Project\.venv\lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 5 members, which is less than n_splits=10.
  warnings.warn(


[48/49] wine_quality_white.tsv.gz: BA=0.2416
[49/49] yeast.tsv.gz: BA=0.3951

Bottom Results (49 datasets):


,dataset,accuracy,macro_f1,balanced_accuracy
0,GAMETES_Epistasis_2_Way_1000atts_0.4H_EDM_1_ED...,0.483125,0.483036,0.483125
1,GAMETES_Epistasis_2_Way_20atts_0.1H_EDM_1_1.ts...,0.496250,0.495443,0.496250
2,GAMETES_Epistasis_2_Way_20atts_0.4H_EDM_1_1.ts...,0.593125,0.593112,0.593125
3,GAMETES_Epistasis_3_Way_20atts_0.2H_EDM_1_1.ts...,0.522500,0.522070,0.522500
4,GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_...,0.544375,0.543146,0.544375
5,GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_...,0.604375,0.604004,0.604375
6,Hill_Valley_with_noise.tsv.gz,0.523102,0.502262,0.523102
7,Hill_Valley_without_noise.tsv.gz,0.482673,0.482546,0.482565
8,analcatdata_aids.tsv.gz,0.440000,0.356618,0.440000
9,analcatdata_asbestos.tsv.gz,0.771084,0.753786,0.751175


In [21]:
# Rank and save bottom results
rank_metric = "accuracy"

bottom_ranked_df = bottom_results_df.sort_values(by=rank_metric, ascending=False).reset_index(drop=True)
bottom_ranked_df.insert(0, "rank", bottom_ranked_df.index + 1)

display(bottom_ranked_df)
bottom_ranked_df.to_csv("bottom_mlp_results_ranked.csv", index=False)
print("Bottom ranked results saved to: bottom_mlp_results_ranked.csv")

,rank,dataset,accuracy,macro_f1,balanced_accuracy
0,1,poker.tsv.gz,0.995366,0.680158,0.634753
1,2,pendigits.tsv.gz,0.990357,0.990372,0.990384
2,3,page_blocks.tsv.gz,0.965101,0.776825,0.702943
3,4,letter.tsv.gz,0.945100,0.944885,0.944827
4,5,new_thyroid.tsv.gz,0.916279,0.874836,0.883175
5,6,segmentation.tsv.gz,0.907359,0.906209,0.907359
6,7,connect_4.tsv.gz,0.805971,0.597680,0.593993
7,8,iris.tsv.gz,0.773333,0.762548,0.773333
8,9,analcatdata_asbestos.tsv.gz,0.771084,0.753786,0.751175
9,10,sonar.tsv.gz,0.764423,0.762839,0.762376


Bottom ranked results saved to: bottom_mlp_results_ranked.csv
